# Proceso de automatizacion contable

In [1]:
# importar librerias a usar
import pandas as pd
from clase_reportes import ReportClass
import numpy as np
import tkinter as tk
from tkinter import filedialog
import json
import re
import os
rc = ReportClass()


In [2]:
ruta = rc.validar_ruta()
ruta_contabilidad = ruta / 'data' / 'contabilidad'
df_base = rc.consolidar_carpeta(ruta_carpeta=ruta_contabilidad / 'odoo' )

df_base = df_base.rename(columns={'Cuenta': 'Cuenta Origen'})
df_base['Cuenta'] = df_base['Cuenta Origen'].str.split(' ', regex=True).str[0]
df_base['Nombre cuenta'] = df_base['Cuenta Origen'].str.split(' ', regex=True).str[1:].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
df_base['N1'] = df_base['Cuenta Origen'].astype(str).str[0]
df_base['N2'] = df_base['Cuenta Origen'].astype(str).str[:2]
df_base['N3'] = df_base['Cuenta Origen'].astype(str).str[:4]

# Define la columna nivel
df_base['Nivel']  =np.where(df_base['N2']=='41', 'Ingreso Operativo',
        np.where(df_base['N2']=='42', 'Otros ingresos',
                np.where(df_base['N2']=='52', 'Gastos operacionales',
                        np.where(df_base['N2']=='53', 'Gastos No Operacionales',
                                    np.where(df_base['N2']=='61', 'Costo directo de ventas', 
                                            'Revisar'
                                    )
                        )
                )
        )
)




df_niveles =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='niveles')
df_concepto_unico =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='cuentas_concepto_uni')
df_concepto =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='concepto_depende_cc')
df_cc =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='CC')
df_concepto_doble =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='doble concepto')



Buscando archivos con extensión '.xlsx' en: C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\contabilidad\odoo
  - Archivo 'adicionaljunio.xlsx' leído correctamente.
  - Archivo 'conlolidado4.xlsx' leído correctamente.
  - Archivo 'consolidado5.xlsx' leído correctamente.
  - Archivo 'consolidado6.xlsx' leído correctamente.
  - No se pudo leer el archivo '~$adicionaljunio.xlsx'. Error: [Errno 13] Permission denied: 'C:\\Users\\Dataa\\Desktop\\VENTAS\\VENTA MENSUAL\\data\\contabilidad\\odoo\\~$adicionaljunio.xlsx'
Concatenando todos los archivos...
¡Consolidación completada!


In [3]:
influencer =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='INFLUENCER')

In [5]:



df_base['N3'] = df_base['N3'].astype(int)
df_base['N3'] = df_base['N3'].astype(int)
df_base['Cuenta'] = df_base['Cuenta'].astype(int)

df_base_merge = df_base.merge(df_niveles, left_on='N3', right_on='cuenta', how='left').drop(columns='cuenta')

def extraer_clave(diccionario_str):
    if pd.isna(diccionario_str):
        return None
    try:
        diccionario = json.loads(diccionario_str)
        return list(diccionario.keys())[0]
    except Exception:
        return None

df_base_merge = df_base_merge.rename(columns={'Distribución analítica': 'Distribución analítica ori'})

df_base_merge['Distribución analítica'] = df_base_merge['Distribución analítica ori'].apply(extraer_clave)


# Ajustes manuales de asignación de centro de costo y concepto
df_base_merge['N1'] = df_base_merge['N1'].astype(str)
df_base_merge['N2'] = df_base_merge['N2'].astype(str)
df_base_merge['N3'] = df_base_merge['N3'].astype(str)

df_base_merge.loc[
    (df_base_merge['N3'] == '4135'),
    'Distribución analítica', 
] = '6'



df_base_merge.loc[
    (df_base_merge['N1'] == '6') & (df_base_merge['Distribución analítica ori'].isna()),
    'Distribución analítica'
] =  '6'


df_base_merge.loc[
    (df_base_merge['N2'] == '42') & (df_base_merge['Distribución analítica ori'].isna()),
    'Distribución analítica'
] = '6'



df_base_merge.loc[(df_base_merge['Distribución analítica'].isna()) & 
            (df_base_merge['Número'].str.startswith('BNK')) &
                (df_base_merge['Cuenta Origen'].isin(['530515001 COMISIONES','530505002 GRAVAMEN CUATRO POR MIL', '530505001 CUOTA DE MANEJO']))
            , 'Distribución analítica'
            ] = '7'


df_base_merge.loc[(df_base_merge['Distribución analítica'].isna()) & 
            (df_base_merge['Número'].str.startswith('BNK')) &
                (df_base_merge['Cuenta Origen'].isin(['539595001 AJUSTE A MILES']))
            , 'Distribución analítica'
            ] = '6' 


In [7]:
df_base_merge.loc[(df_base_merge['Distribución analítica'].isna()) & 
            (df_base_merge['Número'].str.startswith('STJ')) &
            (~df_base_merge['Contacto'].isin(influencer['Contacto'].unique().tolist()))
            , 'Distribución analítica'
            ] = '6'  # validar si es clientre cc ==comercial  o infulerce cc== marketing ==

df_base_merge.loc[(df_base_merge['Distribución analítica'].isna()) & 
            (df_base_merge['Número'].str.startswith('STJ')) &
            (df_base_merge['Contacto'].isin(influencer['Contacto'].unique().tolist()))
            , 'Distribución analítica'
            ] = '4'  # validar si es clientre cc ==comercial  o infulerce cc== marketing ==


df_cc['cc'] = df_cc['cc'].astype(str)

df_base_merge = df_base_merge.merge(df_cc[['cc','Nombre Cencosto', 'ADM/VTAS','Origen' ]],
                                    left_on='Distribución analítica', right_on='cc', how='left').drop(columns='cc')



In [9]:

df_base_merge = df_base_merge.merge(df_concepto_unico, on='Cuenta', how='left')


In [10]:

df_concepto['Nombre Cencosto'] = df_concepto['Nombre Cencosto'].str.upper().str.strip()

df_base_merge['Nombre Cencosto'] = df_base_merge['Nombre Cencosto'].str.upper().str.strip()


In [11]:

df_base_merge = df_base_merge.merge(df_concepto, on=['Cuenta','Nombre Cencosto' ], how='left')
# Crea la columna conceto con base la los coceptos unicos y los que necesitan cc



In [13]:
df_base_merge['Concepto_uni'] = df_base_merge['Concepto_uni'].fillna('Sin datos')

In [14]:
df_base_merge['Concepto_cc'] = df_base_merge['Concepto_cc'].fillna('Sin datos')

In [15]:

# df_base_merge['Concepto'] = np.where(df_base_merge['Concepto_uni'].isna(), df_base_merge['Concepto_cc'], df_base_merge['Concepto_uni'])
df_base_merge = df_base_merge.reset_index(drop=True)
df_base_merge['Concepto'] = np.where(
    df_base_merge['Concepto_uni']=='Sin datos',
    df_base_merge['Concepto_cc'],
    df_base_merge['Concepto_uni']
)


In [17]:

df_concepto_doble['id'] = df_concepto_doble['Nombre Cencosto'] + df_concepto_doble['Cuenta'].astype(str)
df_concepto_doble = df_concepto_doble.drop_duplicates(subset=['id'], keep='first')
df_base_merge = df_base_merge.merge(df_concepto_doble, on=['Cuenta','Nombre Cencosto'],  how='left')


In [18]:

# Verifica las cuentas que no tienen concepto
df_cuentas = df_base_merge[df_base_merge['Concepto'].isna()][['Cuenta','Nombre Cencosto', 'Distribución analítica']]
df_cuentas = df_cuentas.drop_duplicates(subset=['Cuenta', 'Nombre Cencosto',], keep='first')
df_concepto = df_concepto.drop_duplicates(subset=['Cuenta', 'Nombre Cencosto'])


In [20]:

df_concepto_doble = df_concepto_doble.drop_duplicates(subset=['Cuenta'])

df_cuentas= df_cuentas.merge(df_concepto, on=['Cuenta','Nombre Cencosto'], how='left').merge(df_concepto_doble, on=['Cuenta','Nombre Cencosto'], how='left')
df_cuentas = df_cuentas.fillna('Sin datos')


In [21]:

df_cuentas['Estado Cuenta'] = np.where(
    (df_cuentas['Concepto_cc']=="Sin datos") & (df_cuentas['Concepto_doble']== 'Sin datos'),
    'Cuenta Nueva',
    np.where(
        df_cuentas['Concepto_doble'].notna(),
        'Cuenta Doble Concepto',
        'Revisar'
    )
)


In [22]:

df_cuentas = df_cuentas[['Cuenta', 'Nombre Cencosto','Estado Cuenta', 'Distribución analítica']]

# Elimina las columnas que no son necesarias
df_base_merge = df_base_merge.drop(columns=['Concepto_uni', 'Concepto_cc'])

max_date = df_base_merge['Fecha'].max()
min_date = df_base_merge['Fecha'].min()
min_date.strftime('%d-%m-%Y')
ruta_base = ruta_contabilidad / 'base' / f'base_{min_date.strftime('%d-%m-%Y')}_{max_date.strftime('%d-%m-%Y')}.csv'
df_base_merge.to_csv(ruta_base, sep=";", index=False, encoding='utf-8', decimal=',')

centros_no_re = df_base_merge[(df_base_merge['Nombre Cencosto'].isna())&
            (~df_base_merge['Distribución analítica'].isna())
            ][['Distribución analítica ori','Distribución analítica', 'Nombre Cencosto' ]].drop_duplicates()
# Centros de costo mal clasificados
cc_corregir = df_base_merge[df_base_merge['Distribución analítica ori'].fillna('').str.count(':')>1]

# Genera el archivo de los casos sin centro de costos
sin_cc = df_base_merge[df_base_merge['Distribución analítica'].isna()]
sin_cc.to_excel(ruta_contabilidad / 'sin_cc.xlsx', index=False)

# Genera el archivo con los errores
with pd.ExcelWriter(ruta_contabilidad / 'correciones.xlsx', engine='openpyxl') as writer:
    sin_cc.to_excel(writer, index=False, sheet_name='Sin CC')
    cc_corregir.to_excel(writer, index=False, sheet_name='Corregir CC')
    centros_no_re.to_excel(writer, index=False, sheet_name='CC_no_registrados')
    df_cuentas.to_excel(writer, index=False, sheet_name='Cuentas sin concep')


# Crea un archivo para cada persona de contabilidad
digitadores = sin_cc['Creado por'].unique()
ruta_errores = ruta_contabilidad / 'correcciones'
dicc = {}
for i in digitadores:
    base = sin_cc[sin_cc['Creado por']==i]
    base.to_csv(ruta_errores / f'{i}.csv', index=False, sep=';', decimal=',', encoding='utf-8')
    dicc[i] = f'{i}.csv'


In [23]:
df_base_consol =  rc.consolidar_carpeta(extension='csv', encoding='utf-8', sep=';', decimal=',', ruta_carpeta= ruta_contabilidad / 'base')
# pd.read_csv(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\contabilidad\base\base_ene_jun_2025.csv",encoding='utf-8', sep=';')
df_base_consol.to_csv(ruta_contabilidad / 'base_consolidada.csv', encoding='utf-8', sep=';', decimal=',', index=False)


Buscando archivos con extensión '.csv' en: C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\contabilidad\base


c:\Users\Dataa\Desktop\repo_analitica\clase_reportes.py:85: DtypeWarning: Columns (13,27,34,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_completa, sep=sep, encoding=encoding, decimal=decimal)


  - Archivo 'base_30-06-2025_24-10-2025.csv' leído correctamente.


c:\Users\Dataa\Desktop\repo_analitica\clase_reportes.py:85: DtypeWarning: Columns (27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_completa, sep=sep, encoding=encoding, decimal=decimal)


  - Archivo 'base_ene_jun_2025.csv' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!


In [25]:

# df_base_consol.to_excel(ruta_contabilidad / 'base_consolidada.xlsx',  index=False)
